In [ ]:
from dataloading import *
from bpetokenizer import *
from model_arch import *
from RRDataset import *

In [ ]:
class Phase3ClassificationModel(nn.Module):
    def __init__(self, base_model, num_classes=13, pad_token_id=0):
        super().__init__()
        self.base_model = base_model
        self.num_classes = num_classes
        self.pad_token_id = pad_token_id
        
        self.classification_head = nn.Linear(self.base_model.cfg.d_model, num_classes)

    def forward(self, idx):
        B, T = idx.shape
        
        x = self.base_model.tok_emb(idx) + self.base_model.pos_emb[:T]
        for block in self.base_model.blocks:
            x = block(x)
        hidden_states = self.base_model.ln_f(x)
        
        mask = (idx != self.pad_token_id).float().unsqueeze(-1)
        sum_embeddings = (hidden_states * mask).sum(dim=1)
        real_token_counts = mask.sum(dim=1)
        
        pooled_output = sum_embeddings / torch.clamp(real_token_counts, min=1e-9)
        
        logits = self.classification_head(pooled_output)
        
        return logits

In [ ]:
def sequence_kd_loss(student_logits, teacher_logits, labels, temperature=2.0, alpha=0.5):
    ce_loss = F.cross_entropy(student_logits, labels)
    student_soft = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_soft = F.softmax(teacher_logits / temperature, dim=-1)
    kl_loss = F.kl_div(student_soft, teacher_soft, reduction="batchmean") * (temperature ** 2)
    return (alpha * ce_loss) + ((1.0 - alpha) * kl_loss)

In [ ]:
label_vocab, cl_train, cl_dev, cl_test, it_train, it_dev, it_test = load_rr()
tokenizer = load_bpe_tokenizer()

combined_train_data = cl_train + it_train
combined_dev_data = cl_dev + it_dev

combined_train_data = (cl_train + it_train)
combined_dev_data = (cl_dev + it_dev)

train_dataset = RRDataset(combined_train_data, tokenizer, label_vocab)
dev_dataset = RRDataset(combined_dev_data, tokenizer, label_vocab)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

In [ ]:
import torch.optim

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

teacher_base = DecoderLM(LMConfig(vocab_size=50260, n_layers=8))
teacher_model = Phase3ClassificationModel(teacher_base, num_classes=13, pad_token_id=0)
teacher_model.load_state_dict(torch.load("checkpoint.pt", map_location=device))
teacher_model.to(device)
teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False

student_base = DecoderLM(LMConfig(vocab_size=50260, n_layers=8))
student_base.load_state_dict(torch.load("dec.pt", map_location=device))

keep_layers = [0, 2, 4, 6]
student_base.cfg.n_layers = len(keep_layers)
student_base.blocks = nn.ModuleList([student_base.blocks[i] for i in keep_layers])

student_model = Phase3ClassificationModel(student_base, num_classes=13, pad_token_id=0)
student_model.classification_head.load_state_dict(teacher_model.classification_head.state_dict())
student_model.to(device)

optimizer = torch.optim.AdamW(student_model.parameters(), lr=4e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
from sklearn.metrics import f1_score

epochs = 10

for epoch in range(epochs):
    student_model.train()
    total_train_loss = 0
    
    for batch_idx, (b_inputs, b_labels) in enumerate(train_loader):
        b_inputs, b_labels = b_inputs.to(device), b_labels.to(device)
        
        optimizer.zero_grad()
        student_logits = student_model(b_inputs)
        with torch.no_grad():
            teacher_logits = teacher_model(b_inputs)
            
        loss = sequence_kd_loss(student_logits, teacher_logits, b_labels)
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
            
    student_model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for b_inputs, b_labels in dev_loader:
            b_inputs, b_labels = b_inputs.to(device), b_labels.to(device)
            
            logits = student_model(b_inputs)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(b_labels.cpu().numpy())
            
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    avg_train_loss = total_train_loss / len(train_loader)
    
    print("Train Loss: ", avg_train_loss, "Val Macro-F1:", macro_f1)

torch.save(student_model.state_dict(), "checkpoint.pt")